# Catanatron RL Baseline setup
This notebook builds a simple baseline RL agent for Settlers of Catan using `catanatron`, `catanatron-gym` and `sb3-contrib`.

## Important Limitations
- **Multi-agent learning**: `catanatron-gym` provides a Gym interface for 1v1 play. The opponent is fixed via configurations at environment creation. To do self-play, we'd need custom pettingzoo environments or alternating opponent configs, but this notebook focuses on a fixed opponent setting.
- **Action Masking**: The Catan action space varies dramatically per turn. We must use `MaskablePPO` from `sb3-contrib` to filter out illegal moves.
- **Sparse Rewards**: The default reward is +1 for a win, -1 for a loss. Catan runs for hundreds of turns, so learning from sparse rewards is challenging.
- **Agent Speed**: Running full Catan trajectories is slow. We establish a modest baseline here that is runnable in a reasonable time, rather than training for convergence.


In [8]:
!pip install -q catanatron catanatron-gym gymnasium "stable-baselines3[extra]" sb3-contrib


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [9]:
import numpy as np
import gymnasium as gym

import catanatron
import catanatron_gym
from catanatron.models.player import Color, RandomPlayer
from catanatron.players.search import VictoryPointPlayer

from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib.common.wrappers import ActionMasker

print("Catanatron version:", getattr(catanatron, '__version__', 'unknown'))


Catanatron version: unknown


## Environment Inspection
First let's instantiate the Catan environment and check the spaces.

In [10]:
env_id = "catanatron_gym:catanatron-v1"
env = gym.make(env_id)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

obs, info = env.reset()
print("Observation shape:", obs.shape)
print("Keys in info dict upon reset:", info.keys())


Observation space: Box(0.0, 95.0, (614,), float64)
Action space: Discrete(290)
Observation shape: (614,)
Keys in info dict upon reset: dict_keys(['valid_actions'])


## Action & Observation Format
Notice that `catanatron-v1` gives a 614-D vector observation (representing the board state, player cards, resources, etc.) and a Discrete action space of 290. 
Not all 290 actions are valid at any given time. Valid actions are provided in `info['valid_actions']`. Let's create an action masker wrapper for `sb3-contrib`.

In [11]:
def mask_fn(env: gym.Env) -> np.ndarray:
    """
    A helper for valid action masking. 
    ActionMasker from sb3-contrib requires a function that takes the env and returns a boolean array of length `env.action_space.n`.
    """
    # catanatron env unwrapped has a handy get_valid_actions() method
    valid_actions = env.unwrapped.get_valid_actions()
    mask = np.zeros(env.action_space.n, dtype=bool)
    mask[valid_actions] = True
    return mask

# Now wrap it
env = ActionMasker(env, mask_fn)


## Training Setup
We'll train a `MaskablePPO` baseline. We configure the environment to play against a built-in `RandomPlayer` (color red). The agent plays as Blue.


In [12]:
# Re-create environment with the random opponent explicitly to be sure
# Default vps_to_win is 10.
train_env = gym.make("catanatron_gym:catanatron-v1", config={"enemies": [RandomPlayer(Color.RED)]})
train_env = ActionMasker(train_env, mask_fn)

# Instantiate Maskable PPO
model = MaskablePPO(
    MaskableActorCriticPolicy,
    train_env, 
    verbose=1, 
    learning_rate=1e-3, 
    n_steps=2048,
    batch_size=64
)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


## Training Loop
Let's run a short training loop. Feel free to increase `total_timesteps` for better performance.

In [15]:
%%time
print("Training started...")
model.learn(total_timesteps=100000)
print("Training finished.")


Training started...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 505      |
|    ep_rew_mean     | 0.5      |
| time/              |          |
|    fps             | 972      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 488         |
|    ep_rew_mean          | 0.75        |
| time/                   |             |
|    fps                  | 1063        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.015530296 |
|    clip_fraction        | 0.115       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.461      |
|    explained_variance   | 0.823       |
|    learn

## Evaluation Loop
We'll evaluate against simple baselines provided by Catanatron (`RandomPlayer` and `VictoryPointPlayer`). `VictoryPointPlayer` acts greedily to maximize VPs in the short term.


In [18]:
def evaluate_agent(eval_env, agent, episodes=10):
    rewards = []
    lengths = []
    
    for _ in range(episodes):
        obs, info = eval_env.reset()
        done = False
        episode_reward = 0
        steps = 0
        
        while not done:
            # MaskablePPO predict takes the mask! So we provide it explicitly during evaluation
            action_masks = mask_fn(eval_env)
            action, _states = agent.predict(obs, action_masks=action_masks, deterministic=True)
            
            obs, reward, terminated, truncated, info = eval_env.step(action)
            episode_reward += reward
            steps += 1
            done = terminated or truncated
            
        rewards.append(episode_reward)
        lengths.append(steps)
        
    avg_reward = np.mean(rewards)
    # Reward is +1 for win, -1 for loss. Win rate = (avg_reward + 1) / 2
    win_rate = (avg_reward + 1) / 2.0
    avg_len = np.mean(lengths)
    
    return win_rate, avg_len

# 1. Eval vs Random
print("Evaluating against RandomPlayer...")
eval_env_random = gym.make("catanatron_gym:catanatron-v1", config={"enemies": [RandomPlayer(Color.RED)]})
eval_env_random = ActionMasker(eval_env_random, mask_fn)
win_rate_random, len_random = evaluate_agent(eval_env_random, model, episodes=20)
print(f"Vs Random -> Win Rate: {win_rate_random * 100:.1f}% | Avg Episode Length: {len_random:.1f} steps")

# 2. Eval vs VictoryPointPlayer
print("\nEvaluating against VictoryPointPlayer...")
eval_env_vp = gym.make("catanatron_gym:catanatron-v1", config={"enemies": [VictoryPointPlayer(Color.RED)]})
eval_env_vp = ActionMasker(eval_env_vp, mask_fn)
win_rate_vp, len_vp = evaluate_agent(eval_env_vp, model, episodes=100)
print(f"Vs VictoryPointPlayer -> Win Rate: {win_rate_vp * 100:.1f}% | Avg Episode Length: {len_vp:.1f} steps")


Evaluating against RandomPlayer...
Vs Random -> Win Rate: 50.0% | Avg Episode Length: 426.6 steps

Evaluating against VictoryPointPlayer...
Vs VictoryPointPlayer -> Win Rate: 37.5% | Avg Episode Length: 350.0 steps


## Results & Next Steps
We've successfully constructed a pipeline using `MaskablePPO` capable of legal random play and gradient updates against built-in opponents.

**Note on performance**: If you trained for a very short time (e.g., 10,000 steps) and see a win rate of 0% (Average Reward -1.0), this is completely expected. Catan requires complex forward planning and thousands of timesteps just to finish a single game. The random-like initial policy will usually lose against `RandomPlayer` simply because 10,000 steps is not nearly enough to learn to prioritize victory points over dead-end actions.

**Next steps for improvement**:
1. **Longer Training**: Scale training to `1_000_000` or `10_000_000` timesteps. Use `sb3`'s `VecEnv` to parallelize environment stepping.
2. **Network Architecture**: The state is a 614-D vector, but there is graph structure and topology. Using a Graph Neural Network (GNN) on the tile/node definitions would likely provide stronger inductive biases.
3. **Self-Play**: Catan is multi-agent. PPO trains against a fixed distribution of opponents (here, just Random). Using PettingZoo or a wrapper to dynamically swap the opponent policy with past versions of the agent (self-play) will prevent overfitting to weak baselines.
4. **Reward Shaping**: The sparse `+/- 1` reward signal is difficult to learn from. Supplying intermediate rewards for building roads, harvesting resources, or getting VPs will speed up training.
5. **Action Hierarchy**: Our discrete 290-action space represents a flattened action tree (e.g. Build -> Road -> Location). A parameterized action space or a multi-step autoregressive action head might capture the structure better.
